# 实验8.3 昇腾香橙派嵌入式智能机器人实验

> 香橙派开发板（昇腾 310B NPU）· ROS2 · SLAM 建图 · 自主定位与路径导航 · 具身智能

本 notebook 是**实验8 嵌入式智能机器人实验**的配套教学文档，基于已预先调通并搭建好 ROS2 与昇腾平台的**香橙派 Orange Pi AI Pro 开发板**，聚焦机器人运动控制、SLAM 建图、自主定位与路径导航的完整流程。

**定位说明**：本 notebook 在**昇腾香橙派开发板**上运行/查阅，以**真机操作**为主线。各代码单元格的作用是**把真机终端命令以可读方式罗列并解释含义**，方便学生边看边在终端执行；部分以 `numpy/matplotlib` 写的“仿真辅助”单元格用于在板子上直观理解原理（差速运动学、栅格建图、AMCL 粒子收敛、A* 规划），不影响真机流程。

**配套代码**：本实验从 `exp8_test_code/src` 工作空间中提取了与各步骤配套的关键源码（launch 启动脚本、配置参数、URDF 模型、底盘/雷达驱动源码），统一放在 `./code/` 目录下，按 `launch / config / urdf / src_control / src_lslidar` 分类。文档中凡出现 `【代码】` 字样，即指该命令/参数在 `code/` 下对应文件，可对照阅读。

---

## 目录

1. 实验概述
2. 实验原理
3. 功能包与 launch 启动脚本调用关系
4. 实验内容与步骤（一步步启动 → 建图 → 定位 → 导航）
5. 实验结果与分析
6. 关键问题探究
7. 常见问题与故障排查
8. 拓展任务（选做）
9. 附录：配置文件与命令速查
10. 课后练习

---

## 1. 实验概述

### 1.1 实验背景

机器人是人工智能走向物理世界的最佳载体。前面的实验我们已在云沙箱和开发板上完成了模型训练、ONNX→OM 转换与 NPU 推理部署，但那些实验中的“智能”都停留在静态的图片与数据上。本实验把昇腾算力平台真正装进一台移动机器人：在已预先调通并搭建好 ROS2 与昇腾平台的香橙派开发板基础环境上，聚焦机器人控制与 SLAM 建图中的关键问题点——不再耗时于环境安装，而是把全部精力投入到**“让机器人动起来、认得出环境、找得到路”**这三个核心命题上。

实验用车为三轮层叠式结构的智能小车（crobot）：

- **底层驱动板**：负责电机驱动与编码器采集
- **中间主控板**：运行 FreeRTOS 完成里程计解算与通信调度
- **最上层昇腾香橙派开发板**：运行 ROS2 系统与雷达导航算法

![实验样机实物图](./images/robot_real_photo.jpeg)

图：实验样机实物——顶层蓝色电路板即昇腾香橙派开发板（含散热风扇与 GPIO 排针），前部黑色圆盘为 LSLIDAR 激光雷达，两侧为驱动轮与轮胎。

### 1.2 实验目标

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">目标类型</th>
<th style="text-align: left;">内容</th>
</tr>
<tr>
<td style="text-align: left;">知识目标</td>
<td style="text-align: left;">理解两轮差速底盘运动学模型与 <code>/cmd_vel → 电机 → 编码器 → /odom</code> 控制回路；理解 ROS2 DDS 通信机制与 <code>ROS_DOMAIN_ID</code> 隔离作用；理解 <code>slam_toolbox</code> 激光 SLAM 的扫描匹配与位姿图优化原理；理解 AMCL 粒子滤波定位与 Nav2 导航框架（全局规划 + 局部控制 + 恢复行为）的分工</td>
</tr>
<tr>
<td style="text-align: left;">能力目标</td>
<td style="text-align: left;">能够在真机上完成环境就绪验证与整体启动；能够用键盘与 <code>ros2 topic pub</code> 两种方式控制小车运动并读取里程计；能够操作 <code>slam_toolbox</code> 完成室内建图并保存地图；能够在 RViz2 中设置初始位姿、观察 AMCL 粒子收敛并发送导航目标点</td>
</tr>
<tr>
<td style="text-align: left;">素养目标</td>
<td style="text-align: left;">形成“低速、平稳、留余量”的机器人现场操作规范；面对真机问题能区分“软件配置问题”与“物理世界问题”（打滑、漂移、遮挡）</td>
</tr>
</table>

### 1.3 实验环境

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">内容</th>
</tr>
<tr>
<td style="text-align: left;">目标硬件</td>
<td style="text-align: left;">crobot 智能小车：两轮差速底盘，三层结构（驱动板 + 主控板 FreeRTOS + 昇腾香橙派开发板）</td>
</tr>
<tr>
<td style="text-align: left;">激光雷达</td>
<td style="text-align: left;">LSLIDAR N10 串口激光雷达，量程 0.1~12 m</td>
</tr>
<tr>
<td style="text-align: left;">主机平台</td>
<td style="text-align: left;">昇腾香橙派开发板（昇腾 310B4 NPU），运行 ROS2 与导航算法</td>
</tr>
<tr>
<td style="text-align: left;">软件环境</td>
<td style="text-align: left;">ROS2 Humble · CANN · slam_toolbox · Nav2 · RMW = <code>rmw_cyclonedds_cpp</code></td>
</tr>
<tr>
<td style="text-align: left;">工作空间</td>
<td style="text-align: left;"><code>~/crobot_ws_ros2</code>（已 <code>colcon build</code> 编译完成，已写入 <code>~/.bashrc</code>）</td>
</tr>
<tr>
<td style="text-align: left;">PC 端</td>
<td style="text-align: left;">Ubuntu 22.04 + ROS2 Humble，RViz2 可视化、SSH 远程终端；与小车同一局域网且 <code>ROS_DOMAIN_ID</code> 一致</td>
</tr>
<tr>
<td style="text-align: left;">学时</td>
<td style="text-align: left;">4 学时，小组协作实操 + 集中问题研讨</td>
</tr>
</table>

### 1.4 实验学时与组织形式

本实验建议 **4 学时**完成，采用“小组协作实操 + 集中问题研讨”的组织形式，每组 2-3 人为宜：一人负责终端操作、一人负责 RViz2 观察与记录、一人负责安全看护与数据抄录，各步骤轮换角色。建图与导航环节请严格遵守低速操作规范，避免碰撞与打滑。

> **为什么不再把环境搭建作为重点？** 真机环境（ROS2 Humble、雷达驱动、底盘控制、昇腾平台）已预先调通——这恰恰是工程现场的常态：你接手的往往是一套“能跑”的系统，真正的价值在于理解它、用好它、改得动它。步骤一的“就绪验证”会教你用话题与 TF 快速判断系统是否健康，这套“先体检、再动手”的习惯值得带到自己的项目中。

---

## 2. 实验原理

### 2.1 系统架构：三层硬件与 ROS2 软件栈

crobot 小车采用典型的“上下位机”分层架构：

- **下位机**（底层驱动板 + 中间主控板）：实时性要求高的工作——驱动板直接驱动电机并采集编码器脉冲；主控板加载 FreeRTOS，把电机控制、里程计解算、串口通信等任务并发挂起。
- **上位机**（昇腾香橙派开发板）：计算密集型工作——运行 ROS2 系统，依次启动底盘通信节点与激光雷达驱动；根据任务需要进一步拉起 `slam_toolbox`（建图）或 `map_server + Nav2`（导航）。

ROS2 侧核心功能包与职责：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">功能包</th>
<th style="text-align: left;">职责</th>
<th style="text-align: left;">关键产物</th>
</tr>
<tr>
<td style="text-align: left;"><code>crobot_description</code></td>
<td style="text-align: left;">发布机器人 URDF 模型</td>
<td style="text-align: left;">各连杆坐标系与 TF 关系</td>
</tr>
<tr>
<td style="text-align: left;"><code>crobot_control</code></td>
<td style="text-align: left;">底盘串口控制节点</td>
<td style="text-align: left;">接收 <code>/cmd_vel</code>，发布 <code>/odom</code> 与 <code>odom→base_footprint</code> TF</td>
</tr>
<tr>
<td style="text-align: left;"><code>lslidar_driver</code></td>
<td style="text-align: left;">LSLIDAR 雷达串口驱动</td>
<td style="text-align: left;">发布 <code>/scan</code> 激光扫描话题</td>
</tr>
<tr>
<td style="text-align: left;"><code>crobot_slam</code></td>
<td style="text-align: left;">建图配置与 RViz 配置</td>
<td style="text-align: left;"><code>mapper_params_online_async.yaml</code> 参数</td>
</tr>
<tr>
<td style="text-align: left;"><code>crobot_navigation</code></td>
<td style="text-align: left;">导航配置、地图与 RViz 配置</td>
<td style="text-align: left;"><code>nav2_params.yaml</code> 参数、<code>maps/</code> 地图文件</td>
</tr>
<tr>
<td style="text-align: left;"><code>crobot_bringup</code></td>
<td style="text-align: left;">一键整体启动</td>
<td style="text-align: left;">统一拉起模型、底盘、雷达、摄像头四个模块</td>
</tr>
</table>

### 2.2 ROS2 通信机制：节点、话题与 DDS 发现

ROS2 节点不经过中心节点，而是通过 **DDS**（Data Distribution Service）在局域网内自动发现并对等通信。PC 想看到小车话题需满足三个条件：

1. PC 和小车连接**同一局域网**且能互相 `ping` 通
2. 双方使用相同的 `ROS_DOMAIN_ID`（域编号，不同域之间完全隔离）
3. 双方使用兼容的 RMW 实现（本实验统一为 `rmw_cyclonedds_cpp`）

这三个条件都写在 `~/.bashrc` 中，新终端自动生效：

```bash
# PC 与小车两端都执行（以 ROS_DOMAIN_ID=10 为例）
echo "export ROS_DOMAIN_ID=10" >> ~/.bashrc
echo "export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp" >> ~/.bashrc
source ~/.bashrc
```

> **重要**：真机和 Gazebo 仿真使用相同的 `/scan`、`/odom`、`/cmd_vel` 话题名。**绝不能在同一个 `ROS_DOMAIN_ID` 下同时运行真机和仿真**，否则话题会混在一起，RViz 里会出现“两个机器人”的灵异现象——看真机就跑 `crobot_bringup`，跑仿真就只开 PC 端的 Gazebo/SLAM/Nav2，二者选其一。

### 2.3 运动控制链路：从 /cmd_vel 到里程计

运动控制是一条“下行指令 + 上行反馈”的闭环链路，理解每一环的角色是排查一切运动问题的前提：

![运动控制闭环](./images/bringup_modules.png)

- **指令下行**：键盘节点（`teleop_twist_keyboard`）或 Nav2 的 DWB 控制器向 `/cmd_vel` 话题发布 `Twist` 消息（线速度 `linear.x`、角速度 `angular.z`）；`crobot_control` 节点订阅后做两轮差速运动学逆解算，把整车速度分解为左右轮目标转速，经串口 Modbus 寄存器写入驱动板，电机转动。
- **反馈上行**：驱动板读取编码器脉冲，主控板做运动学正解推演，得到底盘相对 `odom` 坐标系的位姿 `(x, y, yaw)`，由 `crobot_control` 发布 `/odom` 话题并广播 `odom→base_footprint` 的 TF 变换——SLAM 与 AMCL 都依赖这条 TF。
- **物理约束**：轮式里程计的前提是“轮子纯滚动”。速度过快会导致轮胎打滑，里程计立刻失真并连锁污染建图与定位——这就是全实验反复强调低速操作（线速度建议 ≤0.2 m/s）的物理原因。
- **停止**：请显式发送一次零速度（键盘按 `K` 键或 `ros2 topic pub` 零 `Twist`），避免底盘节点因“最后一条指令”持续输出。

### 2.4 SLAM 建图：slam_toolbox 的工作原理

本实验使用 `slam_toolbox` 的 **`online_async`（在线异步）**模式：小车一边运动，算法一边把每一帧激光扫描与已有地图做**扫描匹配**（scan matching）求得当前位姿，同时维护一张**位姿图**（pose graph），检测到闭环时做全局图优化，逐步修正累积误差，最终输出分辨率 0.05 m 的**占据栅格地图**（栅格值：占据/空闲/未知）。

**建图质量三条经验判据**（来自真机实践，务必理解而非死记）：

1. **直行才会更新地图，原地旋转不更新**——扫描匹配需要足够的平移视差才能解算出可靠的相对位姿，纯旋转时激光帧间差异过小，算法不做匹配校正。
2. **时刻观察 RViz2 中彩色激光点云与已识别黑色障碍物轮廓是否贴合**，不贴合就慢速调整位姿让其重新贴合。
3. **一旦出现大面积错位，不要试图“救回来”，直接重启建图流程**——错误的位姿图会持续污染后续所有帧。

### 2.5 自主定位与路径规划：AMCL 与 Nav2

有了地图之后，导航要解决“我在哪”和“怎么去”两个问题。

**AMCL**（自适应蒙特卡洛定位）回答“我在哪”：在地图中撒出一堆代表“可能的机器人位姿”的粒子（RViz 中小车周围的红色密集小箭头），用激光观测不断为粒子打分、重采样，粒子云逐渐收拢到真实位姿附近。因此导航前**必须先用 `2D Pose Estimate` 给定大致初始位姿**，否则粒子无从收敛。

**Nav2** 回答“怎么去”，其执行链路为：

- **全局规划**：`NavfnPlanner` 基于全局代价地图用 **Dijkstra 算法**解算一条从当前位姿到目标点的最优参考轨迹（RViz 中的**绿色路径**）。
- **局部控制**：`DWB` 控制器以约 **15 Hz** 在速度空间做动态窗口采样，结合局部代价地图按 `PathAlign`、`PathDist`、`GoalAlign`、`GoalDist` 等评价因子加权打分，输出最优 `/cmd_vel` 给底盘（**蓝色局部路径**）。
- **恢复行为**：当底盘长时间无法动弹时，`behavior_server` 按序调用 `Spin`（原地旋转）、`BackUp`（倒车）等策略帮助脱困——看到机器人“自己转圈”不要急着断电，先看日志判断是否在执行恢复行为。

![Nav2 架构](./images/nav_after_initial_pose.png)

图：设置初始位姿后的导航 RViz——彩色地图为全局代价地图，小车周围更深色部分为局部代价地图，红色密集小箭头即 AMCL 粒子云。

### 2.6 昇腾算力在机器人智能中的角色

本实验的 SLAM 与导航算法本身以 CPU 计算为主，但昇腾香橙派开发板搭载的 NPU 让这台小车具备了“边导航边感知”的升级潜力：摄像头采集的图像可在 NPU 上经 OM 模型完成实时目标检测，检测结果再以 ROS2 话题形式接入导航决策——例如识别人或特定障碍物并动态调整代价地图。这正是“ROS2 + 昇腾平台”组合的完整含义：**ROS2 提供机器人系统的骨架，昇腾提供 AI 感知的肌肉**。本实验先把骨架练好，感知接入作为拓展任务。

---

## 3. 功能包与 launch 启动脚本调用关系

理解“一条命令到底启动了哪些节点”是本实验的基本功。本节把 6 个功能包的 launch 脚本调用关系讲透，并指向 `code/` 目录下的对应文件，方便对照阅读。

### 3.1 功能包一览

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">功能包</th>
<th style="text-align: left;">作用</th>
<th style="text-align: left;">是否使用</th>
</tr>
<tr>
<td style="text-align: left;"><code>crobot_description</code></td>
<td style="text-align: left;">机器人模型包，保存 URDF/mesh 和 RViz 配置，发布机器人模型与 TF</td>
<td style="text-align: left;">使用</td>
</tr>
<tr>
<td style="text-align: left;"><code>crobot_control</code></td>
<td style="text-align: left;">真机底盘控制包，接收 <code>/cmd_vel</code>，串口控制底盘，发布 <code>/odom</code></td>
<td style="text-align: left;">使用</td>
</tr>
<tr>
<td style="text-align: left;"><code>lslidar_driver</code></td>
<td style="text-align: left;">雷神雷达串口驱动包，发布 <code>/scan</code></td>
<td style="text-align: left;">使用</td>
</tr>
<tr>
<td style="text-align: left;"><code>lslidar_msgs</code></td>
<td style="text-align: left;">雷神雷达自定义消息包，供 <code>lslidar_driver</code> 使用</td>
<td style="text-align: left;">使用</td>
</tr>
<tr>
<td style="text-align: left;"><code>crobot_slam</code></td>
<td style="text-align: left;">建图配置包，保存 <code>slam_toolbox</code> 参数和建图 RViz 配置</td>
<td style="text-align: left;">使用</td>
</tr>
<tr>
<td style="text-align: left;"><code>crobot_navigation</code></td>
<td style="text-align: left;">导航配置包，保存 Nav2 参数、地图文件和导航 RViz 配置</td>
<td style="text-align: left;">使用</td>
</tr>
<tr>
<td style="text-align: left;"><code>crobot_bringup</code></td>
<td style="text-align: left;">组合启动包，把模型、底盘、雷达等模块组合启动</td>
<td style="text-align: left;">使用</td>
</tr>
</table>

### 3.2 launch 启动脚本调用关系图

下图展示各 launch 文件之间的**包含（include）关系**——箭头表示“由谁启动了谁”。理解这张图，就能明白为什么“启动 `navigation.launch.py` 一条命令，底盘/雷达/模型/地图/AMCL/规划器全都有了”。

```text
crobot_navigation/navigation.launch.py        ← 一键导航（最顶层）
  ├── crobot_bringup/crobot.launch.py          ← 整机启动
  │     ├── crobot_description/crobot_description.launch.py   ← 机器人模型/TF
  │     ├── crobot_control/crobot_control.launch.py           ← 底盘(/cmd_vel,/odom)
  │     └── lslidar_driver/lslidar_serial.launch.py           ← 雷达(/scan)
  └── crobot_navigation/localization.launch.py ← 地图+AMCL
        ├── nav2_map_server  (map_server)      ← 加载地图
        └── nav2_amcl       (amcl)             ← 粒子滤波定位
  + 直接启动的 Nav2 节点（均加载 params/nav2_params.yaml）：
        controller_server / planner_server /
        behavior_server  / bt_navigator /
        lifecycle_manager_navigation（统一托管上述节点的生命周期）

crobot_slam/slam_toolbox.launch.py             ← 一键建图
  ├── crobot_description/crobot_description.launch.py
  ├── crobot_control/crobot_control.launch.py   （use_base:=true 时）
  ├── lslidar_driver/lslidar_serial.launch.py
  └── slam_toolbox/online_async_launch.py       ← 在线异步建图节点
  + use_base:=false 时改为发布一个静态 odom→base_footprint TF

crobot_slam/save_map.launch.py                 ← 保存地图（一次性命令）
  └── ros2 run nav2_map_server map_saver_cli -f <path>

crobot_navigation/map_server.launch.py         ← 仅加载地图（调试用）
  └── map_server + lifecycle_manager
```

**关键洞察**：`navigation.launch.py` 最顶层一条命令，就把“模型 + 底盘 + 雷达 + 地图 + AMCL + 全局规划 + 局部控制 + 恢复行为”全部拉起。日常建图用 `slam_toolbox.launch.py`，日常导航用 `navigation.launch.py`，排查问题时才退回到 `crobot.launch.py` 或分模块启动。

### 3.3 关键 launch 文件与配置文件解读（对照 `code/` 目录）

下表列出本实验涉及的关键文件，均可在 `./code/` 下找到同名副本，建议逐行对照阅读：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">文件</th>
<th style="text-align: left;"><code>code/</code> 路径</th>
<th style="text-align: left;">作用要点</th>
</tr>
<tr>
<td style="text-align: left;">整机启动</td>
<td style="text-align: left;"><code>code/launch/crobot.launch.py</code></td>
<td style="text-align: left;">聚合 description+control+lidar，参数 <code>base_port</code>/<code>robot_base</code>/<code>laser_port</code></td>
</tr>
<tr>
<td style="text-align: left;">机器人模型</td>
<td style="text-align: left;"><code>code/launch/crobot_description.launch.py</code></td>
<td style="text-align: left;">读 URDF 启动 <code>robot_state_publisher</code> 发布 TF</td>
</tr>
<tr>
<td style="text-align: left;">底盘控制</td>
<td style="text-align: left;"><code>code/launch/crobot_control.launch.py</code></td>
<td style="text-align: left;">启动底盘节点，加载 <code>motor.yaml</code> + <code>robot_base/<type>.yaml</code></td>
</tr>
<tr>
<td style="text-align: left;">雷达驱动</td>
<td style="text-align: left;"><code>code/launch/lslidar_serial.launch.py</code></td>
<td style="text-align: left;">串口雷达节点，发布 <code>/scan</code></td>
</tr>
<tr>
<td style="text-align: left;">建图</td>
<td style="text-align: left;"><code>code/launch/slam_toolbox.launch.py</code></td>
<td style="text-align: left;">聚合模型+底盘+雷达+slam_toolbox</td>
</tr>
<tr>
<td style="text-align: left;">保存地图</td>
<td style="text-align: left;"><code>code/launch/save_map.launch.py</code></td>
<td style="text-align: left;">调 <code>map_saver_cli</code> 一次性保存</td>
</tr>
<tr>
<td style="text-align: left;">导航</td>
<td style="text-align: left;"><code>code/launch/navigation.launch.py</code></td>
<td style="text-align: left;">聚合 bringup+localization+Nav2 全家桶</td>
</tr>
<tr>
<td style="text-align: left;">定位</td>
<td style="text-align: left;"><code>code/launch/localization.launch.py</code></td>
<td style="text-align: left;">map_server + AMCL</td>
</tr>
<tr>
<td style="text-align: left;">URDF 模型</td>
<td style="text-align: left;"><code>code/urdf/edu_robot.urdf</code></td>
<td style="text-align: left;">默认小车模型（base_footprint+两轮+雷达+摄像头）</td>
</tr>
<tr>
<td style="text-align: left;">建图参数</td>
<td style="text-align: left;"><code>code/config/mapper_params_online_async.yaml</code></td>
<td style="text-align: left;">分辨率 0.05、闭环检测、扫描匹配阈值</td>
</tr>
<tr>
<td style="text-align: left;">导航参数</td>
<td style="text-align: left;"><code>code/config/nav2_params.yaml</code></td>
<td style="text-align: left;">DWB 速度上限、代价地图、控制器频率 15Hz</td>
</tr>
<tr>
<td style="text-align: left;">AMCL 参数</td>
<td style="text-align: left;"><code>code/config/amcl_nav2_params.yaml</code></td>
<td style="text-align: left;">粒子数、激光模型、recovery 参数</td>
</tr>
<tr>
<td style="text-align: left;">电机参数</td>
<td style="text-align: left;"><code>code/config/motor.yaml</code></td>
<td style="text-align: left;">编码器线数 3900、PID 周期</td>
</tr>
<tr>
<td style="text-align: left;">2WD 底盘</td>
<td style="text-align: left;"><code>code/config/robot_base_2wd.yaml</code></td>
<td style="text-align: left;">轮半径 0.078、轮距 0.172</td>
</tr>
<tr>
<td style="text-align: left;">底盘源码</td>
<td style="text-align: left;"><code>code/src_control/crobot_control_node.cpp</code> 等</td>
<td style="text-align: left;"><code>/cmd_vel</code> 订阅、运动学解算、串口通信</td>
</tr>
<tr>
<td style="text-align: left;">雷达源码</td>
<td style="text-align: left;"><code>code/src_lslidar/lslidar_driver_node.cc</code> 等</td>
<td style="text-align: left;">串口数据解析、<code>/scan</code> 发布</td>
</tr>
</table>

#### `crobot.launch.py` 做了什么？（`【代码】code/launch/crobot.launch.py`）

这是日常最常用的“一键启动”脚本。它本身**不直接启动任何节点**，而是通过 `IncludeLaunchDescription` 把三个子 launch 串起来：

```python
# 核心逻辑（简化）
IncludeLaunchDescription(crobot_description.launch.py, args={'robot_model': 'edu_robot'})
IncludeLaunchDescription(crobot_control.launch.py,    args={'port_name': '/dev/smart_car', 'robot_base': '2wd'})
IncludeLaunchDescription(lslidar_serial.launch.py,    args={'port_name': '/dev/wheeltec_lidar', 'lidar_name': 'N10'})
```

默认参数：底盘串口 `/dev/smart_car`、底盘类型 `2wd`（两轮差速）、雷达串口 `/dev/wheeltec_lidar`、雷达型号 `N10`。如果你的小车串口设备名不同，可在命令行覆盖：`ros2 launch crobot_bringup crobot.launch.py base_port:=/dev/ttyACM0`。

#### `slam_toolbox.launch.py` 做了什么？（`【代码】code/launch/slam_toolbox.launch.py`）

建图脚本比整机启动多了 slam_toolbox 节点，并用 `use_base` 参数控制是否启动底盘：

- `use_base:=true`（默认）：启动真实底盘，建图时用真实里程计
- `use_base:=false`：不接底盘，仅用雷达建图，此时脚本会发布一个**静态 `odom→base_footprint` TF** 让 SLAM 树完整

slam_toolbox 节点加载 `code/config/mapper_params_online_async.yaml`，关键参数：分辨率 `resolution: 0.05`、最大激光距离 `max_laser_range: 8.0`、闭环检测 `do_loop_closing: true`、地图更新间隔 `map_update_interval: 5.0`。

#### `navigation.launch.py` 做了什么？（`【代码】code/launch/navigation.launch.py`）

导航脚本是最顶层的“全家桶”：先 include `crobot.launch.py`（整机），再 include `localization.launch.py`（地图+AMCL），最后直接启动 4 个 Nav2 节点（`controller_server`/`planner_server`/`behavior_server`/`bt_navigator`），并由 `lifecycle_manager_navigation` 统一托管它们的生命周期（`autostart: True`）。所有 Nav2 节点共享 `code/config/nav2_params.yaml`。

关键参数速览（`nav2_params.yaml`）：
- `controller_server.controller_frequency: 15.0`（DWB 控制频率 15Hz）
- `DWB` 速度上限 `max_vel_x: 0.2`、`max_vel_theta: 1.5`
- `local_costmap` 滚动窗口 6×6 m，分辨率 0.02；`global_costmap` 分辨率 0.05
- `inflation_layer` 膨胀半径 0.25 m，`robot_radius: 0.18`

---

## 4. 实验内容与步骤

本章所有操作在“小车（昇腾香橙派主机）+ PC”双端协同下完成：**小车端命令**在其本机终端或 PC 的 SSH 远程终端中执行（`ssh pi@小车IP`，密码 `pi`），**RViz2 可视化**在 PC 端执行。真机已预配置好环境变量，打开终端即可直接运行 `ros2 launch` 命令，无需手动 `source`。所有终端程序均可按 `Ctrl+C` 终止。

> 请按步骤顺序执行，每步确认预期现象后再进行下一步。

### 步骤一：环境就绪验证与 DDS 通信检查

动手之前先“体检”——确认 PC 与小车的 DDS 通信链路畅通。这一步能提前暴露 90% 的“连不上”问题。

**命令含义速查**：
- `ping <IP>`：测试网络连通性
- `ros2 topic list`：列出当前 ROS2 域内所有话题
- `ros2 topic echo <话题> --once`：打印一帧话题数据后退出
- `ros2 topic hz <话题>`：统计话题发布频率
- `echo $ROS_DOMAIN_ID`：查看当前域编号

In [ ]:
%%bash
# === 真机命令（在 PC 终端执行）===
# 1. 检查网络连通性（IP 以实际小车为准，文档示例为 192.168.10.249）
ping 192.168.10.249

# 2. 检查 ROS2 话题列表
ros2 topic list
#   预期看到: /scan, /odom, /cmd_vel, /tf, /tf_static, /camera/image_raw ...

# 3. 检查特定话题数据
ros2 topic echo /scan --once    # 查看激光雷达数据（应看到 ranges 数组）
ros2 topic echo /odom --once    # 查看里程计数据（应看到 pose 与 twist）
ros2 topic hz /scan             # 查看雷达发布频率（预期 ~10 Hz）
ros2 topic hz /odom             # 查看里程计频率（预期 ~50 Hz）

# 4. 检查 ROS_DOMAIN_ID（PC 和小车应输出相同数字）
echo $ROS_DOMAIN_ID

# 若能看到 /scan、/odom 等话题并能 echo 出数据，说明 DDS 通信已打通。
# 若看不到，按 ping → ROS_DOMAIN_ID → 是否 source（ros2 pkg list | grep crobot）顺序排查。

**排查不通时的三步法**：

```bash
ping 192.168.10.249            # 1. PC 能否 ping 通小车
echo $ROS_DOMAIN_ID           # 2. 两边域编号是否一致
ros2 pkg list | grep crobot   # 3. 当前终端是否 source 了工作空间
```

记录本步 `ros2 topic list` 的完整输出作为实验证据。

### 步骤二：真机整体启动与分模块认知

日常使用推荐**一键整体启动** `crobot_bringup`，它会统一拉起机器人模型、底盘、雷达和摄像头四个模块：

![一键启动的四个模块](./images/bringup_modules.png)

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模块</th>
<th style="text-align: left;">实际启动内容</th>
<th style="text-align: left;">作用</th>
</tr>
<tr>
<td style="text-align: left;">机器人模型</td>
<td style="text-align: left;"><code>crobot_description/crobot_description.launch.py</code></td>
<td style="text-align: left;">发布机器人模型与 TF，让 SLAM/Nav2/RViz 知道各部件坐标关系</td>
</tr>
<tr>
<td style="text-align: left;">底盘</td>
<td style="text-align: left;"><code>crobot_control/crobot_control.launch.py</code></td>
<td style="text-align: left;">接收 <code>/cmd_vel</code>，发布 <code>/odom</code> 与 <code>odom→base_footprint</code> TF</td>
</tr>
<tr>
<td style="text-align: left;">雷达</td>
<td style="text-align: left;"><code>lslidar_driver/lslidar_serial.launch.py</code></td>
<td style="text-align: left;">驱动 LSLIDAR，发布 <code>/scan</code></td>
</tr>
<tr>
<td style="text-align: left;">摄像头</td>
<td style="text-align: left;"><code>crobot_usb_camera/usb_camera.launch.py</code></td>
<td style="text-align: left;">发布 <code>/camera/image_raw</code></td>
</tr>
</table>

`【代码】` 整机启动脚本见 `code/launch/crobot.launch.py`。

In [ ]:
%%bash
# === 真机命令（在小车终端或 SSH 远程终端执行）===
# 一键整体启动（推荐）
ros2 launch crobot_bringup crobot.launch.py

# 如果串口设备名不同，可在命令行覆盖参数：
# ros2 launch crobot_bringup crobot.launch.py base_port:=/dev/ttyACM0 laser_port:=/dev/ttyUSB0

# 启动后另开终端验证：
ros2 topic list                       # 应出现 /scan /odom /cmd_vel /camera/image_raw
ros2 topic echo /scan --once          # 确认雷达有数据

**分模块启动（排查问题时使用）**——把四个模块分四个终端启动，便于单独看日志定位问题：

In [ ]:
%%bash
# === 分模块启动（每个命令开一个终端）===
# 终端1：机器人模型（发布 URDF 与 TF）
ros2 launch crobot_description crobot_description.launch.py

# 终端2：底盘（接收 /cmd_vel，发布 /odom 与 TF）
ros2 launch crobot_control crobot_control.launch.py

# 终端3：雷达（发布 /scan），随后用 ros2 topic echo /scan --once 确认
ros2 launch lslidar_driver lslidar_serial.launch.py

# 终端4：摄像头（发布 /camera/image_raw）
ros2 launch crobot_usb_camera usb_camera.launch.py

# 【代码】对应文件：
#   code/launch/crobot_description.launch.py
#   code/launch/crobot_control.launch.py
#   code/launch/lslidar_serial.launch.py

> **思考**：为什么“机器人模型”这个看似只用于显示的模块，SLAM 和 Nav2 也离不开它？
> 提示：雷达点云要投影到 `map`/`odom` 坐标系，必须知道雷达相对底盘的安装位置——即 TF。`crobot_description` 发布的 URDF 正是 TF 树的来源。

### 步骤三：键盘运动控制与里程计验证

底盘启动后，用键盘控制小车运动：

![键盘控制](./images/keyboard_control.png)

In [ ]:
%%bash
# === 真机命令（另开一个终端）===
# 键盘遥控
ros2 run teleop_twist_keyboard teleop_twist_keyboard

# 按键说明：
#   i: 前进    ,: 后退    k: 停止
#   j: 左转    l: 右转
#   w/e: 增加线速度/角速度    x/c: 降低线速度/角速度

# 请以较低线速度和角速度操作，否则小车打滑会导致里程计不准、建图变差。

也可以用命令行手动发布速度进行**定量测试**（10 Hz 持续发布 0.1 m/s 前进 + 0.1 rad/s 旋转）：

In [ ]:
%%bash
# === 命令行手动发布速度 ===
# 持续发布（-r 10 表示 10Hz）
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0.1, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.1}}" -r 10

# 停止：显式发送一次零速度（或键盘按 K）
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}" --once

# 命令含义：
#   geometry_msgs/msg/Twist  —— 速度消息类型（线速度 linear + 角速度 angular）
#   linear.x  → 前进/后退(m/s)   angular.z → 逆时针/顺时针旋转(rad/s)
#   -r 10     → 以 10Hz 持续发布    --once → 只发一次

运动的同时，在另一终端观察里程计：

In [ ]:
%%bash
# === 观察里程计 ===
ros2 topic echo /odom          # 观察位姿 pose.pose.position 随运动变化
ros2 run tf2_ros tf2_echo odom base_footprint   # 观察 TF 变换

# 动手任务：
#   1. 让小车直行约 1.0 米，用卷尺实测，与 /odom 读数对比（参考误差：未校准 ~3%，校准后 1%~2%）
#   2. 让小车原地旋转一周(2π)，对比偏航角误差（参考值约 5°）

#### 仿真辅助：理解 /cmd_vel 与 /odom 的关系

下面在板子上用 `numpy` 仿真差速底盘的运动控制，模拟真机上 `/cmd_vel → /odom` 的过程，帮助直观理解运动学：

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class DifferentialDriveSim:
    def __init__(self):
        self.x = 0.0; self.y = 0.0; self.theta = 0.0
        self.v = 0.0; self.omega = 0.0
        self.L = 0.28  # 轮距(与 2wd.yaml 中 separation 近似)
        self.odom_trace = [(0, 0, 0)]
    def cmd_vel(self, v, omega):
        self.v = v; self.omega = omega
    def step(self, dt=0.05):
        self.x += self.v * np.cos(self.theta) * dt
        self.y += self.v * np.sin(self.theta) * dt
        self.theta += self.omega * dt
        self.odom_trace.append((self.x, self.y, self.theta))
    def get_odom(self):
        return self.x, self.y, self.theta

robot = DifferentialDriveSim()
commands = [
    (0.2, 0.0, 40),    # 前进
    (0.15, 0.3, 30),   # 左转前进
    (0.2, 0.0, 40),    # 前进
    (0.15, -0.3, 30),  # 右转前进
    (0.0, 0.0, 10),    # 停止
]
print('=== 仿真 /cmd_vel -> /odom ===')
for v, w, n in commands:
    robot.cmd_vel(v, w)
    for _ in range(n):
        robot.step()
    x, y, th = robot.get_odom()
    print(f'cmd_vel(v={v:.2f}, w={w:.2f}) -> odom(x={x:.3f}, y={y:.3f}, theta={np.degrees(th):.1f}deg)')

trace = np.array(robot.odom_trace)
plt.figure(figsize=(8, 6))
plt.plot(trace[:, 0], trace[:, 1], 'b-', linewidth=2)
plt.plot(trace[0, 0], trace[0, 1], 'go', markersize=12, label='起点')
plt.plot(trace[-1, 0], trace[-1, 1], 'r^', markersize=12, label='终点')
plt.xlabel('X (m)'); plt.ylabel('Y (m)')
plt.title('差速底盘里程计轨迹仿真（模拟 /odom）')
plt.legend(); plt.grid(True, alpha=0.3); plt.axis('equal')
plt.tight_layout()
plt.savefig('./images/odom_trace_sim.png', dpi=150, bbox_inches='tight')
plt.show()

### 步骤四：SLAM 实时建图

建图前确认机器人模型、底盘、雷达三个模块已启动（步骤二的一键启动已包含），然后启动 `slam_toolbox` 在线异步建图。

`【代码】` 建图启动脚本见 `code/launch/slam_toolbox.launch.py`，建图参数见 `code/config/mapper_params_online_async.yaml`。

In [ ]:
%%bash
# === 真机命令（在小车终端执行）===
# 方式一：用 crobot_slam 封装的一键建图（推荐，自动拉起模型+底盘+雷达+slam_toolbox）
ros2 launch crobot_slam slam_toolbox.launch.py

# 方式二：在已启动 bringup 的基础上，单独启动 slam_toolbox
ros2 launch slam_toolbox online_async_launch.py \
    use_sim_time:=false \
    slam_params_file:=$HOME/crobot_ws_ros2/src/crobot_slam/params/slam_toolbox/mapper_params_online_async.yaml

# 雷达串口不是默认名时可覆盖：
# ros2 launch crobot_slam slam_toolbox.launch.py laser_port:=/dev/ttyACM0
# 不接底盘仅用雷达建图：
# ros2 launch crobot_slam slam_toolbox.launch.py use_base:=false

**在 PC 端打开建图 RViz**：

In [ ]:
%%bash
# === PC 端命令 ===
rviz2 -d $HOME/crobot_ws_ros2/src/crobot_slam/rviz/slam_create_map.rviz

# 另开终端启动键盘控制，操纵小车缓慢移动建图
ros2 run teleop_twist_keyboard teleop_twist_keyboard

![建图过程](./images/slam_rviz_building.png)

图：slam_toolbox 建图过程——RViz2 中地图随小车运动在线拓展。浅灰色区域为已探索可通行空间，黑色栅格为墙体与家具边缘等障碍物，彩色散点为当前 `/scan` 激光点云在 `map` 坐标系下的投影。

![建图完成效果](./images/slam_complete_effect.png)

图：建图完成效果——激光点云与栅格地图障碍物轮廓基本贴合。

**建图操作要领**（对应 2.4 节原理，违反任何一条都会直接反映在地图质量上）：

1. 低速直行、慢速转弯，**不要快速原地旋转**
2. **原地旋转不会更新地图，直行才会更新**
3. 时刻注意彩色激光点云轮廓与黑色障碍物轮廓是否贴合，不贴合就慢速调整位姿
4. 若出现大面积错位，**重新启动建图流程**（不要试图补救）

建图过程中终端若出现偶发的匹配警告信息可忽略，不影响建图。

#### 仿真辅助：理解占据栅格地图构建

下面仿真一个简化的 SLAM 建图过程，展示栅格地图如何随机器人运动逐步构建：

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

grid_size = 20
grid = np.ones((grid_size, grid_size)) * 0.5  # 未知
true_walls = set()
for i in range(grid_size):
    true_walls.add((0, i)); true_walls.add((grid_size-1, i))
    true_walls.add((i, 0)); true_walls.add((i, grid_size-1))
for i in range(5, 12): true_walls.add((8, i))
for i in range(3, 8): true_walls.add((i, 14))
path = [(2,2),(2,5),(2,8),(2,12),(2,16),(5,16),(5,12),(5,8),(5,5),(5,2),
        (10,2),(12,5),(12,10),(12,16),(16,16),(16,2)]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
steps_to_show = [5, 11, len(path)-1]
for step_idx, (rx, ry) in enumerate(path):
    for dx, dy in [(0,1),(0,-1),(1,0),(-1,0)]:
        for d in range(1, grid_size):
            nx, ny = rx+dx*d, ry+dy*d
            if (nx, ny) in true_walls:
                grid[nx, ny] = 1.0
                for dd in range(1, d):
                    grid[rx+dx*dd, ry+dy*dd] = 0.0
                break
    if step_idx in steps_to_show:
        ax = axes[steps_to_show.index(step_idx)]
        cmap = plt.cm.colors.ListedColormap(['white', 'gray', 'black'])
        ax.imshow(grid, cmap=cmap, origin='lower')
        ax.plot(ry, rx, 'b^', markersize=12)
        ax.set_title(f'建图步骤 {step_idx+1}', fontsize=13)
plt.suptitle('SLAM 建图过程仿真：地图随机器人运动逐步构建', fontsize=14)
plt.tight_layout()
plt.savefig('./images/slam_building_sim.png', dpi=150, bbox_inches='tight')
plt.show()
print('白色=空闲, 灰色=未知, 黑色=占据(墙壁)')

### 步骤五：地图保存与质量检查

对地图满意后，用 `nav2_map_server` 的 `map_saver_cli` 保存（**保持建图节点运行中**执行）：

![保存地图](./images/slam_terminal_info.png)

In [ ]:
%%bash
# === 真机命令（保持建图节点运行，另开终端执行）===
ros2 run nav2_map_server map_saver_cli \
    -f $HOME/crobot_ws_ros2/src/crobot_navigation/maps/crobot_room \
    --free 0.10 --occ 0.65 --mode trinary

# 保存新版本地图时改文件名即可，如 crobot_room_20260710
# ros2 run nav2_map_server map_saver_cli \
#     -f $HOME/crobot_ws_ros2/src/crobot_navigation/maps/crobot_room_20260710 \
#     --free 0.10 --occ 0.65 --mode trinary

# 命令含义：
#   -f <路径>     → 输出文件名前缀（生成同名 .yaml + .pgm）
#   --free 0.10   → 占据概率 < 0.10 判为空闲(白)
#   --occ  0.65   → 占据概率 > 0.65 判为占据(黑)
#   --mode trinary→ 三值化(白/灰/黑)

保存后生成同名 `.yaml`（地图元数据：分辨率、原点、阈值）与 `.pgm`（栅格图像）两个文件。打开 `.pgm` 检查：墙体轮廓是否连续清晰、有无重影错位；打开 `.yaml` 确认 `resolution` 为 `0.05`。

`.yaml` 文件示例：

```yaml
image: crobot_room.pgm
resolution: 0.050000
origin: [-5.0, -5.0, 0.0]   # 地图原点在 map 坐标系下的位姿
occupied_thresh: 0.65        # 与 --occ 对应
free_thresh: 0.10            # 与 --free 对应
negate: 0
```

`【代码】` 保存地图的 launch 封装见 `code/launch/save_map.launch.py`。

### 步骤六：自主定位（AMCL）

导航前确认机器人模型、底盘、雷达已启动，然后启动完整 Nav2（加载刚保存的地图）。

`【代码】` 导航启动脚本见 `code/launch/navigation.launch.py`，定位脚本见 `code/launch/localization.launch.py`，AMCL 参数见 `code/config/amcl_nav2_params.yaml`。

In [ ]:
%%bash
# === 真机命令（在小车终端执行）===
# 方式一：用 crobot_navigation 封装的一键导航（推荐）
ros2 launch crobot_navigation navigation.launch.py

# 方式二：在已启动 bringup 的基础上，单独启动 Nav2（需指定地图与参数）
ros2 launch nav2_bringup bringup_launch.py \
    use_sim_time:=false \
    map:=$HOME/crobot_ws_ros2/src/crobot_navigation/maps/crobot_room.yaml \
    params_file:=$HOME/crobot_ws_ros2/src/crobot_navigation/config/nav2_real.yaml

# === PC 端打开导航 RViz ===
rviz2 -d $HOME/crobot_ws_ros2/src/crobot_navigation/rviz/navigation.rviz

![刚打开 RViz 的报错](./images/nav_rviz_initial.png)

图：刚打开 RViz2 时左侧有报错条目、终端持续输出提示信息——这是**正常的**，因为 AMCL 还不知道小车在哪里。完成下一步“设置初始位姿”后即进入正常导航。

**设置初始位姿**：点击工具栏 `2D Pose Estimate`，对照小车在房间中的实际位置与朝向，在地图上相应位置按下并拖拽出绿色箭头。

![2D Pose Estimate 设置初始位姿](./images/nav_2d_pose_estimate.png)

设置好初始位姿后：彩色地图为**全局代价地图**，小车周围更深色的彩色部分为**局部代价地图**，小车周围红色密集小箭头即 **AMCL 粒子云**。

![设置初始位姿后](./images/nav_after_initial_pose.png)

此时激光点云与地图障碍物轮廓可能不太贴合——用键盘慢速移动或缓慢转圈，让点云与轮廓贴合，红色粒子会逐渐收拢到小车中心附近。**粒子收拢后才允许发送目标点**（这是铁律，原因见问题 3）。

![AMCL 粒子收敛](./images/nav_amcl_particles.png)

#### 仿真辅助：AMCL 粒子收敛过程

下面仿真 AMCL 粒子滤波的收敛过程，帮助理解真机上的定位原理：

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
true_pos = np.array([8.0, 8.0])
n_particles = 300
particles = np.column_stack([
    np.random.normal(7, 1.5, n_particles),
    np.random.normal(7, 1.5, n_particles),
    np.random.normal(0.3, 0.5, n_particles)])
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for step in range(30):
    weights = np.exp(-((particles[:, 0]-true_pos[0])**2 +
                       (particles[:, 1]-true_pos[1])**2) / 2.0)
    weights /= weights.sum()
    if step in [0, 10, 25]:
        ax = axes[[0, 10, 25].index(step)]
        ax.scatter(particles[:, 0], particles[:, 1], c='red', s=8, alpha=0.4, label='粒子')
        ax.plot(true_pos[0], true_pos[1], 'b^', markersize=15, label='真实位姿')
        est = np.average(particles[:, :2], weights=weights, axis=0)
        ax.plot(est[0], est[1], 'g*', markersize=15, label='估计位姿')
        ax.set_title(f'AMCL 步骤 {step}', fontsize=13)
        ax.legend(fontsize=9); ax.set_xlim(3, 13); ax.set_ylim(3, 13)
    indices = np.random.choice(n_particles, n_particles, p=weights)
    particles = particles[indices]
    particles[:, 0] += np.random.randn(n_particles) * 0.2
    particles[:, 1] += np.random.randn(n_particles) * 0.2
    particles[:, 2] += np.random.randn(n_particles) * 0.1
est = np.average(particles[:, :2], weights=weights, axis=0)
print(f'真实位置: ({true_pos[0]}, {true_pos[1]})')
print(f'估计位置: ({est[0]:.2f}, {est[1]:.2f})')
print(f'定位误差: {np.linalg.norm(est - true_pos):.3f} m')
print('粒子从初始散布逐渐收敛到真实位姿附近。')
plt.suptitle('AMCL 粒子滤波定位收敛过程仿真', fontsize=14)
plt.tight_layout()
plt.savefig('./images/amcl_sim.png', dpi=150, bbox_inches='tight')
plt.show()

### 步骤七：目标点导航与路径规划

粒子收敛后，点击工具栏 `2D Goal Pose`，在地图上选择目标点按下并拖拽出绿色箭头（箭头方向即目标朝向）。

![发送导航目标点](./images/nav_2d_goal_pose.png)

松开鼠标后可以看到规划出的**绿色全局路径**（NavfnPlanner）与**蓝色局部路径**（DWB），小车沿路径自主驶向目标。

![导航执行路径](./images/nav_execution_paths.png)

![导航终端日志](./images/nav_terminal_log.png)

图：终端也会输出导航日志（规划耗时、到达判定等），作为导航是否成功的参考信息。

**导航过程三项观察记录**：
1. 终端的导航日志输出（规划耗时、到达判定等）
2. 在预定路径上临时放置一个小障碍物（如纸箱），观察局部代价地图实时映射障碍、DWB 控制器平稳绕行的过程
3. 让小车多次往返不同目标点，记录终点驻泊偏差（实际停靠位置与目标点的距离）

若小车长时间卡住不动，观察 `behavior_server` 是否触发 `Spin`/`BackUp` 恢复行为并把日志记录下来。至此，**“运动控制 → 建图 → 定位 → 导航”**的完整链路全部打通。

---

## 5. 实验结果与分析

### 5.1 运动控制性能分析

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">测试项</th>
<th style="text-align: left;">测试方法</th>
<th style="text-align: left;">参考结果</th>
</tr>
<tr>
<td style="text-align: left;">电机响应</td>
<td style="text-align: left;">串口下发靶速，观测转速稳定时间</td>
<td style="text-align: left;">线速度 20 cm/s 以内约 200 ms 稳定，转速误差 ≤2%</td>
</tr>
<tr>
<td style="text-align: left;">直线里程计精度</td>
<td style="text-align: left;">下发 1.0 m 直线，卷尺实测对比</td>
<td style="text-align: left;">平均相对误差约 3%（纯轮式里程计合理范围）</td>
</tr>
<tr>
<td style="text-align: left;">旋转里程计精度</td>
<td style="text-align: left;">原地转向 2π，对比偏航角</td>
<td style="text-align: left;">误差约 5°</td>
</tr>
<tr>
<td style="text-align: left;">校准后精度</td>
<td style="text-align: left;">下发 <code>set_correction_factor</code></td>
<td style="text-align: left;">累积误差降至 1%~2%</td>
</tr>
<tr>
<td style="text-align: left;">控制链路延迟</td>
<td style="text-align: left;">发送指令到电机动作的时延</td>
<td style="text-align: left;">稳定在约 80 ms</td>
</tr>
</table>

### 5.2 建图质量分析

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">指标</th>
<th style="text-align: left;">参考值</th>
</tr>
<tr>
<td style="text-align: left;"><code>/scan</code> 频率</td>
<td style="text-align: left;">~10 Hz</td>
</tr>
<tr>
<td style="text-align: left;"><code>/odom</code> 频率</td>
<td style="text-align: left;">~50 Hz</td>
</tr>
<tr>
<td style="text-align: left;">每帧扫描点</td>
<td style="text-align: left;">~1800</td>
</tr>
<tr>
<td style="text-align: left;">地图分辨率</td>
<td style="text-align: left;">0.05 m</td>
</tr>
<tr>
<td style="text-align: left;">3 m 直线漂移</td>
<td style="text-align: left;"><0.1 m</td>
</tr>
</table>

### 5.3 定位与导航性能

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">指标</th>
<th style="text-align: left;">参考值</th>
</tr>
<tr>
<td style="text-align: left;">AMCL 收敛时间</td>
<td style="text-align: left;">2~3 秒</td>
</tr>
<tr>
<td style="text-align: left;">线速度上限</td>
<td style="text-align: left;">0.2 m/s</td>
</tr>
<tr>
<td style="text-align: left;">角速度上限</td>
<td style="text-align: left;">0.5 rad/s</td>
</tr>
<tr>
<td style="text-align: left;">DWB 控制频率</td>
<td style="text-align: left;">~15 Hz</td>
</tr>
<tr>
<td style="text-align: left;">obstacle_layer 探测范围</td>
<td style="text-align: left;">~3 m</td>
</tr>
</table>

### 5.4 从“能跑”到“跑好”：参数与工程权衡

把全实验的因果链串起来会发现一个清晰的传递关系：

```text
速度过快 → 轮胎打滑 → 里程计漂移 → 建图错位 / 定位发散 → 导航失败
```

链路上每个环节都对应一个可操作的控制量：操作速度（≤0.2 m/s）、里程计校准（`set_correction_factor`）、建图阈值（`--free`/`--occ`）、AMCL 初始位姿质量、DWB 速度上限与评价因子权重。**机器人工程的功夫不在某个算法多精妙，而在能把整条链路每一环的误差都压在下一环能容忍的范围之内。**

从平台视角看，这套 SLAM + 导航负载完全运行在香橙派 CPU 上且仍有余量，昇腾 NPU 算力处于“待命”状态——这为接入视觉感知（拓展任务）预留了充足空间，体现了“ROS2 + 昇腾”平台面向真实机器人应用的扩展性。

---

## 6. 关键问题探究

### 问题 1：真机与仿真为什么不能共用同一个 ROS_DOMAIN_ID？

真机和 Gazebo 仿真都会发布 `/scan`、`/odom`、`/cmd_vel` 等同名话题。若同域同时运行，RViz 和 SLAM 会同时收到两套数据，出现“两个机器人”模型叠加、地图被两组点云交替污染、Nav2 把速度指令发给了谁也分不清的灵异现象。工程做法：二者选其一，或用不同域 ID 隔离（代价是看不到对方话题）。

### 问题 2：为什么建图时“直行才更新地图、原地旋转不更新”？

扫描匹配需要足够的平移视差才能解算出可靠的相对位姿，纯旋转时激光帧间差异过小，算法不做匹配校正。如果环境是一条两侧完全对称的长走廊，直行时还会因特征区分度不足出现退化。

### 问题 3：为什么必须先用 2D Pose Estimate 给定初始位姿？

AMCL 本质上是“在已知地图中做局部位姿跟踪”而非全局定位，初始粒子按给定位姿高斯撒布。不给初始位姿，粒子无从收敛；粒子没收敛就发目标点几乎必然导航失败。

### 问题 4：map_saver 的 --free 0.10 --occ 0.65 改了会怎样？

这两个阈值是对栅格占据概率做二值化：小于 `--free` 判为空闲(白)，大于 `--occ` 判为占据(黑)，之间为未知(灰)。调宽灰区会让墙体变细、未知区变大；设反（`--free` > `--occ`）则地图逻辑混乱无法用于导航。

### 问题 5：/cmd_vel 到电机动作的约 80ms 延迟由哪些环节构成？

DDS 传输 + 节点解算 + 串口 Modbus 通信 + 电机电气响应。可用 `ros2 topic delay/hz`、串口日志时间戳、电机阶跃响应分别测量。80 ms 指令延迟与 DWB 15 Hz（约 67 ms）控制周期同量级，高速导航时会引入跟踪滞后风险。

### 问题 6：如何把昇腾 NPU 推理接入 ROS2 导航链路？（设计题）

订阅 `/camera/image_raw`，用实验 5/6 的 ONNX→OM 与 AscendCL 链路在 NPU 上运行目标检测模型，检测结果以 ROS2 话题发布，Nav2 侧通过 `obstacle_layer` 消费（代价地图支持订阅自定义数据源）。画出节点/话题拓扑图，分析 NPU 推理相比 CPU 推理对系统实时性的改善。

---

## 7. 常见问题与故障排查

遇到异常时请保持冷静：**先看终端日志，再对照下表逐项排查**；分模块启动（步骤二）是定位问题模块的最有效手段。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">现象</th>
<th style="text-align: left;">可能原因</th>
<th style="text-align: left;">排查与解决</th>
</tr>
<tr>
<td style="text-align: left;">PC 端看不到小车话题</td>
<td style="text-align: left;">不在同一局域网 / <code>ROS_DOMAIN_ID</code> 不一致 / 未 source</td>
<td style="text-align: left;"><code>ping</code> 小车 IP；两边 <code>echo $ROS_DOMAIN_ID</code> 对比；<code>ros2 pkg list | grep crobot</code> 确认环境</td>
</tr>
<tr>
<td style="text-align: left;">RViz 中出现两套机器人数据，地图混乱</td>
<td style="text-align: left;">真机与 Gazebo 仿真同域同时运行</td>
<td style="text-align: left;">二选一：看真机只跑 <code>crobot_bringup</code>，跑仿真只在 PC 端启动</td>
</tr>
<tr>
<td style="text-align: left;"><code>/scan</code> 无数据</td>
<td style="text-align: left;">雷达串口未连接 / 驱动未启动</td>
<td style="text-align: left;">检查雷达 USB/串口接线；单独启动 <code>lslidar_serial.launch.py</code> 看日志；<code>ros2 topic echo /scan --once</code> 验证</td>
</tr>
<tr>
<td style="text-align: left;">键盘控制小车不动</td>
<td style="text-align: left;">底盘节点未启动 / 串口异常 / 速度为 0</td>
<td style="text-align: left;">确认 <code>crobot_control</code> 终端无报错；按 <code>w/e</code> 确认速度倍率不是 0；<code>ros2 topic echo /cmd_vel</code> 确认消息已发出</td>
</tr>
<tr>
<td style="text-align: left;">里程计误差明显偏大</td>
<td style="text-align: left;">速度过快打滑 / 未做参数校准</td>
<td style="text-align: left;">低速重测；下发 <code>set_correction_factor</code> 校准参数</td>
</tr>
<tr>
<td style="text-align: left;">建图点云与轮廓大面积错位</td>
<td style="text-align: left;">快速旋转 / 急停导致匹配失败</td>
<td style="text-align: left;">不要补救，<code>Ctrl+C</code> 重启建图流程；后续保持低速直行、慢速转弯</td>
</tr>
<tr>
<td style="text-align: left;">地图原地旋转不更新</td>
<td style="text-align: left;">算法机制如此（非故障）</td>
<td style="text-align: left;">直行才会触发扫描匹配与地图更新，见 2.4 节</td>
</tr>
<tr>
<td style="text-align: left;">RViz 打开后左侧报错、终端刷提示</td>
<td style="text-align: left;">尚未设置初始位姿（正常现象）</td>
<td style="text-align: left;">用 <code>2D Pose Estimate</code> 给定初始位姿后即正常</td>
</tr>
<tr>
<td style="text-align: left;">发送目标点后小车不动</td>
<td style="text-align: left;">粒子未收敛 / 目标点在障碍物上 / 恢复行为中</td>
<td style="text-align: left;">等粒子收拢后再发点；确认目标点在白色空闲区；查看 <code>behavior_server</code> 日志</td>
</tr>
<tr>
<td style="text-align: left;"><code>colcon build</code> 失败</td>
<td style="text-align: left;">依赖未安装</td>
<td style="text-align: left;"><code>rosdep install --from-paths src --ignore-src -r -y</code></td>
</tr>
</table>

---

## 8. 拓展任务（选做）

完成核心实验内容后，鼓励学有余力的小组挑战以下拓展任务：

1. **一键启动与体检脚本**：编写一个 shell 或 Python 脚本，自动完成“ping 小车 → 检查话题 → 分模块启动 → 健康报告”全流程，输出结构化检查结果。
2. **里程计系统标定**：直行 1.0 m 重复 5 次、原地 2π 旋转重复 5 次，记录里程计读数并拟合校准系数，下发 `set_correction_factor` 后复测，给出校准前后的误差对比表。
3. **建图参数调优**：研究 `mapper_params_online_async.yaml` 中的关键参数（分辨率、匹配窗口、最小平移/旋转更新阈值等），修改 1-2 个参数重新建图，对比地图质量与 CPU 占用。
4. **动态避障压力测试**：在导航路径上动态摆放不同尺寸障碍物，记录局部代价地图映射延迟、DWB 绕行轨迹与是否触发恢复行为；调大/调小 `obstacle_layer` 探测范围观察差异。
5. **昇腾感知接入（衔接实验 5/6）**：把实验 6 训练的 YOLO OM 模型部署到小车——订阅 `/camera/image_raw`，用 AscendCL 在 NPU 上推理，检测结果以话题发布并在 RViz 可视化检测框。评估 NPU 推理帧率相比 CPU 的提升，思考问题 6 中接入代价地图的方案。

---

## 9. 附录：配置文件与命令速查

### 附录A 关键配置文件说明

本实验“少装环境、多调参数”的设计，意味着读懂三个关键配置文件比背命令更重要（均在 `code/config/` 下有副本）：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">文件</th>
<th style="text-align: left;">位置</th>
<th style="text-align: left;">关键参数</th>
</tr>
<tr>
<td style="text-align: left;"><code>mapper_params_online_async.yaml</code></td>
<td style="text-align: left;"><code>crobot_slam/params/slam_toolbox/</code></td>
<td style="text-align: left;">分辨率 <code>resolution: 0.05</code>、<code>max_laser_range: 8.0</code>、闭环检测 <code>do_loop_closing: true</code>、最小更新距离 0.5 m / 0.5 rad</td>
</tr>
<tr>
<td style="text-align: left;"><code>nav2_params.yaml</code></td>
<td style="text-align: left;"><code>crobot_navigation/params/</code></td>
<td style="text-align: left;">DWB <code>max_vel_x: 0.2</code>/<code>max_vel_theta: 1.5</code>、<code>controller_frequency: 15.0</code>、膨胀半径 0.25、<code>robot_radius: 0.18</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>amcl_nav2_params.yaml</code></td>
<td style="text-align: left;"><code>crobot_navigation/params/amcl/</code></td>
<td style="text-align: left;">粒子数 500~5000、<code>likelihood_field</code> 激光模型、<code>max_beams: 60</code>、recovery 参数</td>
</tr>
<tr>
<td style="text-align: left;"><code>motor.yaml</code></td>
<td style="text-align: left;"><code>crobot_control/config/</code></td>
<td style="text-align: left;">编码器线数 <code>count_per_rev: 3900</code>、<code>pid_interval: 50</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>robot_base_2wd.yaml</code></td>
<td style="text-align: left;"><code>crobot_control/config/robot_base/</code></td>
<td style="text-align: left;">轮半径 <code>0.078</code>、轮距 <code>separation: 0.172</code></td>
</tr>
</table>

> **学习方法**：打开这些文件逐行阅读，把每个参数与实验中观察到的现象对应起来——能解释“这个参数影响了哪一步的什么现象”，就真正掌握了这套系统。

### 附录B ROS2 常用命令速查

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">命令</th>
<th style="text-align: left;">作用</th>
<th style="text-align: left;">使用场景</th>
</tr>
<tr>
<td style="text-align: left;"><code>ros2 launch crobot_bringup crobot.launch.py</code></td>
<td style="text-align: left;">一键启动真机全部模块</td>
<td style="text-align: left;">整体启动</td>
</tr>
<tr>
<td style="text-align: left;"><code>ros2 topic list</code> / <code>ros2 topic echo <话题> --once</code></td>
<td style="text-align: left;">查看话题列表 / 打印一帧数据</td>
<td style="text-align: left;">通信检查、各步验证</td>
</tr>
<tr>
<td style="text-align: left;"><code>ros2 topic hz <话题></code></td>
<td style="text-align: left;">统计话题发布频率</td>
<td style="text-align: left;">验证 <code>/scan</code> ~10Hz、<code>/odom</code> ~50Hz</td>
</tr>
<tr>
<td style="text-align: left;"><code>ros2 topic pub /cmd_vel ... -r 10</code></td>
<td style="text-align: left;">以 10Hz 发布速度指令</td>
<td style="text-align: left;">定量运动测试</td>
</tr>
<tr>
<td style="text-align: left;"><code>ros2 run teleop_twist_keyboard teleop_twist_keyboard</code></td>
<td style="text-align: left;">键盘遥控小车</td>
<td style="text-align: left;">建图操控</td>
</tr>
<tr>
<td style="text-align: left;"><code>ros2 launch slam_toolbox online_async_launch.py ...</code></td>
<td style="text-align: left;">启动在线异步建图</td>
<td style="text-align: left;">实时建图</td>
</tr>
<tr>
<td style="text-align: left;"><code>ros2 run nav2_map_server map_saver_cli -f <路径> ...</code></td>
<td style="text-align: left;">保存地图（.yaml + .pgm）</td>
<td style="text-align: left;">地图保存</td>
</tr>
<tr>
<td style="text-align: left;"><code>ros2 launch nav2_bringup bringup_launch.py ...</code></td>
<td style="text-align: left;">启动完整 Nav2 导航</td>
<td style="text-align: left;">定位与导航</td>
</tr>
<tr>
<td style="text-align: left;"><code>rviz2 -d <配置文件></code></td>
<td style="text-align: left;">按指定配置打开可视化</td>
<td style="text-align: left;">建图/导航 RViz</td>
</tr>
<tr>
<td style="text-align: left;"><code>ssh pi@<小车IP></code></td>
<td style="text-align: left;">PC 端远程登录小车终端（密码 pi）</td>
<td style="text-align: left;">所有小车端操作</td>
</tr>
<tr>
<td style="text-align: left;"><code>ros2 node list</code> / <code>ros2 run tf2_ros tf2_echo odom base_footprint</code></td>
<td style="text-align: left;">查看节点 / 查看 TF</td>
<td style="text-align: left;">调试</td>
</tr>
</table>

### 附录C 在 PC 端通过 SSH 打开小车终端

当小车与 PC 连接至同一局域网后，在 PC 端打开终端：

```bash
ssh pi@小车IP地址   # 例如：ssh pi@192.168.10.249
# 接着输入密码：pi
```

成功进入小车终端后，本 notebook 中所有“真机命令”都可以在这个 SSH 终端中执行，无需在小车旁边接显示器键盘。

---

## 10. 课后练习

请根据本节实验内容完成以下题目进行自测。

**第1题**（单选题）本实验的目标硬件平台是？


- A. 树莓派
- B. 昇腾香橙派开发板（昇腾 310B4 NPU）
- C. NVIDIA Jetson
- D. Arduino


In [ ]:
q1 = ''  # 填入你的选项，如 'B'
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')

**第2题**（单选题）crobot小车采用几层硬件架构？


- A. 两层
- B. 三层
- C. 四层
- D. 单层


In [ ]:
q2 = ''  # 填入你的选项，如 'B'
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')

**第3题**（单选题）PC想看到小车话题需满足的条件不包括？


- A. 同一局域网且能ping通
- B. 相同的ROS_DOMAIN_ID
- C. 兼容的RMW实现
- D. 相同的Python版本


In [ ]:
q3 = ''  # 填入你的选项，如 'B'
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')

**第4题**（单选题）本实验SLAM建图使用的工具是？


- A. Cartographer
- B. slam_toolbox online_async
- C. Gmapping
- D. ORB-SLAM


In [ ]:
q4 = ''  # 填入你的选项，如 'B'
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')

**第5题**（单选题）建图时为什么原地旋转不更新地图？


- A. 算法bug
- B. 扫描匹配需要平移视差才能解算位姿
- C. 雷达不工作
- D. 里程计停止


In [ ]:
q5 = ''  # 填入你的选项，如 'B'
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')

**第6题**（单选题）AMCL定位前必须先做什么？


- A. 启动摄像头
- B. 用2D Pose Estimate给定初始位姿
- C. 保存地图
- D. 编译工作空间


In [ ]:
q6 = ''  # 填入你的选项，如 'B'
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')

**第7题**（单选题）map_saver的--free 0.10 --occ 0.65参数分别表示什么？


- A. 起点和终点
- B. 空闲概率阈值和占据概率阈值
- C. 地图分辨率和原点
- D. 膨胀半径和探测范围


In [ ]:
q7 = ''  # 填入你的选项，如 'B'
print(f'第7题答案已记录：{q7}' if q7 else '请填入答案并运行本单元格')

**第8题**（单选题）本实验建议的最大线速度是？


- A. 0.5 m/s
- B. 0.2 m/s
- C. 1.0 m/s
- D. 0.05 m/s


In [ ]:
q8 = ''  # 填入你的选项，如 'B'
print(f'第8题答案已记录：{q8}' if q8 else '请填入答案并运行本单元格')

**第9题**（单选题）Nav2中DWB控制器的控制频率约为？


- A. 5 Hz
- B. 15 Hz
- C. 50 Hz
- D. 100 Hz


In [ ]:
q9 = ''  # 填入你的选项，如 'B'
print(f'第9题答案已记录：{q9}' if q9 else '请填入答案并运行本单元格')

**第10题**（单选题）/cmd_vel到电机动作的延迟约为？


- A. 10ms
- B. 80ms
- C. 500ms
- D. 1s


In [ ]:
q10 = ''  # 填入你的选项，如 'B'
print(f'第10题答案已记录：{q10}' if q10 else '请填入答案并运行本单元格')

**第11题**（单选题）真机和仿真为什么不能共用同一个ROS_DOMAIN_ID？


- A. 性能问题
- B. 话题同名会混在一起导致数据混乱
- C. 版本不兼容
- D. 安全限制


In [ ]:
q11 = ''  # 填入你的选项，如 'B'
print(f'第11题答案已记录：{q11}' if q11 else '请填入答案并运行本单元格')

**第12题**（单选题）昇腾NPU在本实验中的角色是？


- A. 运行SLAM算法
- B. 为感知类AI任务预留算力
- C. 控制电机
- D. 发布/odom


In [ ]:
q12 = ''  # 填入你的选项，如 'B'
print(f'第12题答案已记录：{q12}' if q12 else '请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path
for candidate in (Path.cwd() / 'answer', Path.cwd() / '08_robot_dev' / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_03 import grade
grade(globals())

---

## 参考资料

- [ROS2 官方文档](https://docs.ros.org/en/rolling/)
- [SLAM Toolbox](https://github.com/SteveMacenski/slam_toolbox)
- [Nav2 导航框架](https://navigation.ros.org/)
- [香橙派 AI Pro 官方文档](http://www.orangepi.cn/)
- [robot_localization (EKF)](https://github.com/cra-ros/robot_localization)
- 本实验配套源码：`exp8_test_code/src`（完整工作空间）、`code/`（精选关键文件）
- 参考文档：`港大-ROS2小车功能与运行说明_20260717.pdf`、`实验8_嵌入式智能机器人实验手册.docx`